In [ ]:
app_fqdn = "mtg.test"
proxy_address = "127.0.0.1"
admin_subdomain = "admin"
admin_password = "E2e-Admin-Pass1"
tenant_identifier = "sample"
tenant_name = "Sample"
owner_display_name = "Owner"
owner_username = "owner"
owner_password = "E2e-Owner-Pass1"
default_result_path = None
close_on_fail = False
transition_timeout = 10000

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# WebMeeting E2E Test - Admin Site

- Test data to prepare: Not required. The admin account is created with ADMIN_INITIAL_PASSWORD at the first start.

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *

await init_pw_context(
    close_on_fail=close_on_fail,
    last_path=default_result_path,
    ignore_https_errors=True,
    browser_args=[f"--host-resolver-rules=MAP {app_fqdn} {proxy_address}, MAP *.{app_fqdn} {proxy_address}"],
)

## Log in to the admin site as admin with the initial password, and verify that the tenant list is shown.

In [ ]:
admin_url = f"https://{admin_subdomain}.{app_fqdn}"

async def _step(page):
    await page.goto(f"{admin_url}/login")
    await page.locator("input[name=username]").fill("admin")
    await page.locator("input[name=password]").fill(admin_password)
    await page.locator("button[type=submit]").click()

    await expect(page).to_have_url(f"{admin_url}/tenant", timeout=transition_timeout)

await run_pw(_step)

## Create the tenant tenant_identifier with its owner account, and verify that the tenant detail is shown.

In [ ]:
import re

async def _step(page):
    await page.goto(f"{admin_url}/tenant/create")
    await page.locator("#identifier").fill(tenant_identifier)
    await page.locator("#name").fill(tenant_name)
    await page.locator("#initialDisplayName").fill(owner_display_name)
    await page.locator("#initialName").fill(owner_username)
    await page.locator("#initialPassword").fill(owner_password)
    await page.locator("#initialPasswordConfirm").fill(owner_password)
    await page.locator("button[type=submit]").click()

    await expect(page).to_have_url(re.compile(f"^{admin_url}/tenant/[0-9]+$"), timeout=transition_timeout)
    await expect(page.locator("body")).to_contain_text(tenant_identifier)

await run_pw(_step)

## Log in to the tenant as the owner, and verify that the home page is shown.

In [ ]:
tenant_url = f"https://{tenant_identifier}.{app_fqdn}"

async def _step(page):
    await page.goto(f"{tenant_url}/login")
    await page.locator("input[name=username]").fill(owner_username)
    await page.locator("input[name=password]").fill(owner_password)
    await page.locator("button[type=submit]").click()

    await expect(page).to_have_url(f"{tenant_url}/home", timeout=transition_timeout)

await run_pw(_step, new_context=True)

Clean up

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}